In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../scripts'))
import paths   # input/output locations: set SN2024VJM_DATA / SN2024VJM_EXTERNAL (see README)
import mpdaf.obj
from astropy import units as u
from astropy.io import fits
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np
from astropy.visualization import ZScaleInterval
%matplotlib widget

redshift = 0.0028
z = redshift

# Fix the astrometry

In [ ]:
ra, dec = 287.357417, -63.833819
x , y = 167, 169
hdul = fits.open(paths.MUSE / 'Object.fits')
header = hdul[1].header
header['CRVAL1'] = ra  # Set RA of the reference point
header['CRVAL2'] = dec # Set Dec of the reference point
header['CRPIX1'] = x   # Set X pixel coordinate of the reference point
header['CRPIX2'] = y   # Set Y pixel coordinate of the reference point
hdul.writeto(paths.MUSE / 'updated_wcs_datacube.fits', overwrite=True)

In [ ]:
hdul[2].data.shape

In [ ]:
def air_to_vaccum(lam):
    return (1+1e-6*(287.6155+1.62887/lam**2+0.01360/lam**4)) * lam

def vaccum_to_air(lam):
    return (1+1e-6*(287.6155+1.62887/lam**2+0.01360/lam**4)) * lam

def vactoair(wave):
    wave2 = wave**2.0
    fact = 1.0 + 2.735182e-4 + 131.4182 / wave2 + 2.76249e8 / (wave2**2.0)
    wave = wave / fact
    return wave

In [ ]:
# air_to_vaccum(6562.79)

In [ ]:
import zap
# Inputs and parameters
input_cube = str(paths.MUSE / 'updated_wcs_datacube.fits')
output_cube = str(paths.MUSE / 'updated_wcs_datacube_cleaned.fits')
mask_width = 10  # Adjust this to control skyline width

# Run ZAP with custom parameters
zap.process(input_cube, outcubefits=str(paths.MUSE / 'Out.fits'),cfwidthSVD=20,cfwidthSP=20,overwrite=True)

print(f"Sky subtraction completed. Output saved to {output_cube}")

In [ ]:
plt.figure()
# Load the MUSE datacube
cube = mpdaf.obj.Cube(str(paths.MUSE / 'Out.fits'))
cube = mpdaf.obj.Cube(str(paths.MUSE / 'updated_wcs_datacube.fits'))

# Define the coordinates of the region (e.g., RA and Dec)
ra, dec = 287.357417, -63.833819  # Replace with your RA/Dec

# Convert world coordinates to pixel indices
pix_coords = cube.wcs.sky2pix([dec,ra],nearest=True,unit=u.deg)
x, y = int(pix_coords[0][0]), int(pix_coords[0][1])

# Extract the spectrum at the specified pixel
spectrum = cube[:, y, x]

# Plot the spectrum and error spectrum if available
import matplotlib.pyplot as plt
wavelength = spectrum.wave.coord()/(1+0.0028)
plt.plot(wavelength, spectrum.data*1e-20, label='Flux')
if hasattr(cube, 'var') and cube.var is not None:
    error_spectrum = np.sqrt(cube.var[:, y, x])
    plt.fill_between(wavelength, (spectrum.data*1e-20 - error_spectrum*1e-20), (spectrum.data*1e-20 + error_spectrum*1e-20), color='gray', alpha=0.5, label='Error')
v = 1200 # km/s
c = 300000 # km/s
plt.axvline(6578*(1-v/c), color='red', linestyle='--', label='C II',alpha=0.5)
plt.axvline(7231*(1-v/c), color='blue', linestyle='--', label='C II',alpha=0.5)
plt.xlabel('Wavelength (Angstrom)')
plt.ylabel('Flux')
plt.title('Extracted Spectrum and Error')
plt.legend()

plt.show()

In [ ]:
import numpy as np

# Prepare the output filename
output_file = 'SN2024vjm_MUSE_spectrum_2024-10-13.txt'

# Open the file for writing
with open(output_file, 'w') as f:
    # Write the header as comments
    f.write('# FITS HEADER INFORMATION\n')
    for key, value in header.items():
        f.write(f'# {key} = {value}\n')
    f.write('#\n')
    f.write('# Wavelength (Angstrom)   Flux\n')

    # Write the spectrum data
    wavelength = spectrum.wave.coord()
    flux = spectrum.data
    for wl, fl in zip(wavelength, flux):
        f.write(f'{wl:.3f}   {fl:.6e}\n')

print(f"Spectrum saved to {output_file}")

In [ ]:
cube[:, y, x]

In [ ]:
H_alpha = 6562.8
# H_alpha = H_alpha -5
lambda_obs = H_alpha * (1 + redshift)
delta_lambda = 3.0  # Width of the range in Ångstrom
index = int(cube.wave.pixel(lambda_obs))
low_index = int(cube.wave.pixel(lambda_obs - delta_lambda))
high_index = int(cube.wave.pixel(lambda_obs + delta_lambda))

# Find the closest wavelength index
h_alpha_image = cube[index, :, :]#.sum(axis=0)

# Extract the data from the MPDAF image
data = h_alpha_image.data

# Crop the image to remove edges with missing data (trim edges by 20 pixels)
edge_trim = 20  # pixels to trim from each edge
y_start = edge_trim
y_end = data.shape[0] - edge_trim
x_start = edge_trim
x_end = data.shape[1] - edge_trim

data_cropped = data[y_start:y_end, x_start:x_end]

# Mask out NaN values
data_masked = np.ma.masked_invalid(data_cropped)

# Apply ZScale to determine display limits
zscale = ZScaleInterval()
vmin, vmax = zscale.get_limits(data_masked.compressed())

# Calculate scale bar parameters
pixel_scale_mas = 25  # mas per pixel
distance_mpc = 7.92   # Mpc
pc_per_mas = distance_mpc * 1e6 / 206265e3  # parsecs per milliarcsecond

# Scale bar length in parsecs and pixels
scale_length_pc = 20  # 10 pc scale bar
scale_length_mas = scale_length_pc / pc_per_mas
scale_length_pixels = scale_length_mas / pixel_scale_mas

# Plot in black and white using Matplotlib with ZScale limits
plt.figure(figsize=(10, 10))
ax = plt.gca()

# Use masked array for plotting to hide invalid regions
im = ax.imshow(
    data_masked, 
    origin='lower', 
    cmap='gray',  # Black-and-white colormap
    vmin=0, 
    vmax=vmax,  # ZScale limits
    interpolation='nearest'
)

# Add scale bar (bottom left corner) - actual horizontal bar
scale_bar_x = 10
scale_bar_y = 20
ax.plot([scale_bar_x, scale_bar_x + scale_length_pixels], [scale_bar_y, scale_bar_y], 
        color='white', lw=4)
# Add scale bar label
ax.text(scale_bar_x + scale_length_pixels/2, scale_bar_y + 1, 
        f'{scale_length_pc} pc', color='white', fontsize=14, 
        ha='center', va='bottom', weight='bold')

# Add compass (top left corner)
compass_x = 30
compass_y = data_cropped.shape[0] - 20  # near top
compass_size = 20  # pixels

# North arrow (vertical line pointing up)
ax.plot([compass_x, compass_x], [compass_y - compass_size, compass_y], 
        color='white', lw=3)
ax.text(compass_x, compass_y + 5, 'N', color='white', fontsize=12, 
        ha='center', va='bottom', weight='bold')

# East arrow (horizontal line pointing left, since astronomical convention)
ax.plot([compass_x - compass_size, compass_x], [compass_y - compass_size, compass_y - compass_size], 
        color='white', lw=3)
ax.text(compass_x - compass_size - 5, compass_y - compass_size, 'E', color='white', fontsize=12, 
        ha='right', va='center', weight='bold')

#Add text near the middle
ax.text(data_cropped.shape[1] // 2-2, data_cropped.shape[0] // 2+25, 'SN 2024vjm', color='white', fontsize=12,
        ha='center', va='center', weight='bold')

# Remove ticks and spines for cleaner look
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

# Set title
# ax.set_title(f"H-alpha Emission (λ = {lambda_obs:.1f} Å)", fontsize=16, weight='bold', pad=20)

plt.tight_layout()
plt.savefig(paths.OUTPUTS / 'MUSE_Halpha.png')
plt.show()

In [ ]:
pc_per_mas*25

In [ ]:
# Debug scale bar calculations
print(f"Scale bar length in pixels: {scale_length_pixels:.2f}")
print(f"Scale bar length in pc: {scale_length_pc}")
print(f"Scale bar length in mas: {scale_length_mas:.2f}")
print(f"Data cropped shape: {data_cropped.shape}")
print(f"Scale bar position: x={scale_bar_x}, y={scale_bar_y}")
print(f"Scale bar end position: x={scale_bar_x + scale_length_pixels:.2f}")

In [ ]:
z = 0.0028

x_center, y_center = 195,275
radius = 70  # Radius of the aperture in pixels

# x_center, y_center = 290,50
# radius = 40  # Radius of the aperture in pixels

s = cube.subcube_circle_aperture((x_center, y_center), radius,unit_center=None, unit_radius=None).sum(axis=(1,2))
wavelength = s.wave.coord()  # Wavelength array
flux = s.data          # Flux array

# Extract error spectrum for the aperture if available
error_spectrum = None
if hasattr(cube, 'var') and cube.var is not None:
    # Sum the variance in the same aperture and axis as the flux
    var_ap = cube.var[:, y_center-radius:y_center+radius+1, x_center-radius:x_center+radius+1].sum(axis=(1,2))
    error_spectrum = np.sqrt(var_ap)

#save the spectrum as an ascii
from astropy.table import Table
Table([wavelength/(1+z),flux*1e-20],names=['WAVE_REST','FLAMBDA_EXTCORR']).write(paths.MUSE / 'spectrum.txt',format='ascii',overwrite=True)


# Plot the spectrum with error as a shaded region
plt.figure(figsize=(8, 5))
plt.plot(wavelength/(1+z), flux*1e-20, label='Extracted Spectrum')
if error_spectrum is not None:
    plt.fill_between(wavelength/(1+z), (flux-error_spectrum)*1e-20, (flux+error_spectrum)*1e-20, color='gray', alpha=0.5, label='Error')
else:
    print("Error spectrum not available.")
#plot a line at the rest wavelength of H-alpha, H-beta, O III, N II and [S II]
plt.axvline(6562.80, color='red', linestyle='--', label='Rest Wavelength of H-alpha',alpha=0.5)
plt.axvline(4862.69, color='red', linestyle='--', label='Rest Wavelength of H-beta',alpha=0.5)
plt.axvline(5008.24, color='green', linestyle='--', label='Rest Wavelength of [OIII]',alpha=0.5)
plt.axvline(4960.30, color='green', linestyle='--', label='Rest Wavelength of [OIII]',alpha=0.5)
plt.axvline(6547.96, color='purple', linestyle='--', label='Rest Wavelength of [NII]',alpha=0.5)
plt.axvline(6583.34, color='purple', linestyle='--', label='Rest Wavelength of [NII]',alpha=0.5)
plt.axvline(6716.31, color='blue', linestyle='--', label='Rest Wavelength of [SII]',alpha=0.5)
plt.axvline(6730.68, color='blue', linestyle='--', label='Rest Wavelength of [SII]',alpha=0.5)
# plt.axvline(4960, color='orange', linestyle='--', label='Rest Wavelength of OIII',alpha=0.5)

# plt.axvline(6414.39, color='black', linestyle='--', label='laser',alpha=0.5)


plt.xlabel(r'Rest Wavelength ($\AA$)')
plt.ylabel(r'Flux [$erg s^{-1} cm^{-2} \AA^{-1}$]')
plt.legend()
plt.title('Spectrum from Circular Aperture')
plt.show()

H_alpha = 6562.8
lambda_obs = H_alpha * (1 + redshift)
delta_lambda = 3.0  # Width of the range in Ångstrom
index = int(cube.wave.pixel(lambda_obs))
low_index = int(cube.wave.pixel(lambda_obs - delta_lambda))
high_index = int(cube.wave.pixel(lambda_obs + delta_lambda))

# Find the closest wavelength index
h_alpha_image = cube[index, :, :]#.sum(axis=0)

# Extract the data from the MPDAF image
data = h_alpha_image.data

# Apply ZScale to determine display limits
zscale = ZScaleInterval()
vmin, vmax = zscale.get_limits(data)

# Plot in black and white using Matplotlib with ZScale limits
plt.figure(figsize=(8, 8))
plt.imshow(
    data, 
    origin='lower', 
    cmap='gray',  # Black-and-white colormap
    vmin=0, 
    vmax=vmax,  # ZScale limits
    interpolation='nearest'
    )
aperture = Circle((x_center, y_center), radius, color='red', fill=False, lw=2)
plt.gca().add_patch(aperture)

plt.colorbar(label='Flux')
plt.title(f"H-alpha")
plt.xlabel('X (pixels)')
plt.ylabel('Y (pixels)')
plt.show()

In [ ]:
from scipy.optimize import curve_fit

s = cube.subcube_circle_aperture((x_center, y_center), radius,unit_center=None, unit_radius=None).sum(axis=(1,2))
wavelength = s.wave.coord()  # Wavelength array
flux = s.data          # Flux array

def poly_fit(wl,flux,n=3):
    pfit = np.polyfit(wl,flux,n)
    return np.polyval(pfit, wl),pfit

plt.figure()
poly,poly_func = poly_fit(wavelength,flux,3)
flux = flux - poly
plt.plot(wavelength/(1+z),flux*1e-20,label='Spectrum')

lines = {
    'H-a':          6562.80,
    'H-b':          4862.69,
    '[OIII]-5008':  5008.24,
    # '[OIII]-4960':  4960.30,
    '[NII]-6547':   6547.96,
    '[NII]-6583':   6583.34,
    '[SII]-6716':   6716.31,
    '[SII]-6732':   6730.68,
}

fluxes = {}

def gaussian(x,A,mu_a,sigma_a,c):
    gaus = A*np.exp(-(x-mu_a)**2/(2.*sigma_a**2))+c
    return gaus

def gaussians2(x,A,mu_a,sigma_a,B,mu_b,sigma_b):
    gaus = A*np.exp(-(x-mu_a)**2/(2.*sigma_a**2))
    gaus += B*np.exp(-(x-mu_b)**2/(2.*sigma_b**2))
    return gaus


for line in lines:
    #fit a gaussian to the line
    line_wl = lines[line]
    #choose wavelengths between 10 angstroms of the line
    mask = (wavelength/(1+z) > line_wl-10) & (wavelength/(1+z) < line_wl+10)
    wl = wavelength[mask]/(1+z)
    a = curve_fit(gaussian,wl,flux[mask],p0=[2e4,line_wl,1,0])
    
    #Calculate the flux of the line
    line_flux = np.trapz((gaussian(wl,*a[0])-a[0][3])*1e-20,x=wl)
    fluxes[line] = line_flux
    print(f'{line} flux: {line_flux:.2e} erg/s/cm^2')
    
    #plot the line
    plt.axvline(line_wl,color='red',linestyle='--',label=line)
    plt.plot(wl,gaussian(wl,*a[0])*1e-20,label=f'{line} fit')

for line in lines:
    #fit a gaussian to the line
    line_wl = lines[line]
    #choose wavelengths between 10 angstroms of the line
    mask = (wavelength/(1+z) > line_wl-10) & (wavelength/(1+z) < line_wl+10)

    wl = wavelength[mask]/(1+z)
    popt, pcov = curve_fit(gaussian,wl,flux[mask],p0=[2e4,line_wl,1,0])
    perr = np.sqrt(np.diag(pcov))  # standard deviation of the fitted parameters

    # Propagate uncertainty in flux via error in amplitude and sigma:
    A, mu, sigma, c = popt
    A_err, mu_err, sigma_err, c_err = perr

    # Approximate flux error: ∆F ≈ F × sqrt((∆A/A)² + (2∆σ/σ)²)
    line_flux = np.trapz((gaussian(wl,*popt)-c)*1e-20,x=wl)
    flux_error = line_flux * np.sqrt((A_err/A)**2 + (2*sigma_err/sigma)**2)
    fluxes[line] = (line_flux, flux_error)
    
    print(f'{line} flux: {line_flux:.2e} erg/s/cm^2')
    
    #plot the line
    plt.axvline(line_wl,color='red',linestyle='--',label=line)
    plt.plot(wl,gaussian(wl,A, mu, sigma, c)*1e-20,label=f'{line} fit')
    
plt.show()


In [ ]:
from numpy import log10, sqrt, log
from scipy.optimize import fsolve

print(fluxes)
OIII, dOIII = fluxes['[OIII]-5008']
Hb, dHb     = fluxes['H-b']
NII, dNII   = fluxes['[NII]-6583']
Ha, dHa     = fluxes['H-a']

# _O3N2 = (fluxes['[OIII]-5008']/(fluxes['H-b'])) / (fluxes['[NII]-6583']/fluxes['H-a'])
_O3N2 = (OIII / Hb) / (NII / Ha)
O3N2 = lambda x: np.log10(_O3N2) - (0.281 - 4.765 * x - 2.268 * x**2)
value     = fsolve(O3N2, -0.1, maxfev=10000, xtol=1e-14)


O3N2_val = log10(_O3N2)

dR_R = sqrt((dOIII/OIII)**2 + (dHb/Hb)**2 + (dNII/NII)**2 + (dHa/Ha)**2)
dO3N2 = (1 / log(10)) * dR_R

def dO3N2dx(x):
    return -4.765 - 2*2.268*x  # Derivative of O3N2 function

dx = dO3N2 / abs(dO3N2dx(value[0]))
metallicity = 10**value[0]
metallicity_error = dx * metallicity * log(10)
print(f'O3N2 value: {O3N2_val:.4f} ± {dO3N2:.4f}')
print(f'Metallicity: {metallicity:.4f} ± {metallicity_error:.4f}')

In [ ]:
import numpy as np
from scipy.optimize import fsolve

OIII, dOIII = fluxes['[OIII]-5008']
Hb,   dHb   = fluxes['H-b']
NII,  dNII  = fluxes['[NII]-6583']
Ha,   dHa   = fluxes['H-a']

R = (OIII/Hb) / (NII/Ha)
O3N2 = np.log10(R)

O0 = 8.69  # pivot used by the calibration (adjust if your paper uses a different one)

# Model written in terms of X_abs = 12+log(O/H)
def model_residual(X_abs):
    x = X_abs - O0                 # shift to the calibration’s x
    return O3N2 - (0.281 - 4.765*x - 2.268*x**2)

X = fsolve(model_residual, 8.5, maxfev=10000, xtol=1e-14)[0]  # X ≡ 12+log(O/H)

# Error on O3N2 from fluxes
dR_R = np.sqrt((dOIII/OIII)**2 + (dHb/Hb)**2 + (dNII/NII)**2 + (dHa/Ha)**2)
dO3N2 = dR_R / np.log(10)

# Derivative d(O3N2)/dX for propagation
def dO3N2_dX(X_abs):
    x = X_abs - O0
    return -(4.765 + 2*2.268*x)

sigma_X_stat = dO3N2 / abs(dO3N2_dX(X))

# (Recommended) add intrinsic calibration scatter in quadrature (set from your chosen calibration)
sigma_int = 0.10  # dex, example
sigma_X = np.sqrt(sigma_X_stat**2 + sigma_int**2)

print(f"(12+log O/H) = {X:.3f} ± {sigma_X:.3f} dex")

# If you also want linear O/H (note the asymmetry if you Monte-Carlo it; below is simple transform)
OH = 10**(X - 12)
# For a quick symmetric approximation in linear space (OK for small σ): 
OH_err = OH * np.log(10) * sigma_X
print(f"O/H ≈ {OH:.3e} ± {OH_err:.3e}")

In [ ]:
import numpy as np
from scipy.optimize import fsolve

# -------- Curti+17 calibration metadata --------
O0 = 8.69  # solar pivot used by Curti+17 (Allende Prieto et al. 2001)

CALIBS = {
    # log10( ( [OIII]5007/Hb ) / ( [NII]6584/Ha ) )
    "O3N2": {
        "coeffs": [0.281, -4.765, -2.268, 0.0, 0.0],  # c0..c4
        "sigma_int_dex": 0.09,
        "range": (7.6, 8.85),
        "ratio_fn": lambda f: np.log10( (f['[OIII]-5008'][0]/f['H-b'][0]) / (f['[NII]-6583'][0]/f['H-a'][0]) ),
        "ratio_err_fn": lambda f: (
            # error on log10(ratio) from flux uncertainties, assuming independence
            np.sqrt( (f['[OIII]-5008'][1]/f['[OIII]-5008'][0])**2 +
                     (f['H-b'][1]/f['H-b'][0])**2 +
                     (f['[NII]-6583'][1]/f['[NII]-6583'][0])**2 +
                     (f['H-a'][1]/f['H-a'][0])**2 ) / np.log(10)
        ),
        "name": "O3N2"
    },

    # log10( [NII]6584 / Ha )
    "N2": {
        "coeffs": [-0.489, 1.513, -2.554, -5.293, -2.867],
        "sigma_int_dex": 0.10,
        "range": (7.6, 8.85),
        "ratio_fn": lambda f: np.log10( f['[NII]-6583'][0] / f['H-a'][0] ),
        "ratio_err_fn": lambda f: (
            np.sqrt( (f['[NII]-6583'][1]/f['[NII]-6583'][0])**2 +
                     (f['H-a'][1]/f['H-a'][0])**2 ) / np.log(10)
        ),
        "name": "N2"
    }
}

def invert_curti17(fluxes, which="O3N2", x0=8.5):
    meta = CALIBS[which]
    c0, c1, c2, c3, c4 = meta["coeffs"]

    # measured log(ratio) and its error
    logR = meta["ratio_fn"](fluxes)
    dlogR = meta["ratio_err_fn"](fluxes)

    # polynomial for logR as a function of X = 12+log(O/H)
    def poly(X):
        x = X - O0
        return c0 + c1*x + c2*x**2 + c3*x**3 + c4*x**4

    # residual to solve: logR - poly(X) = 0
    X = fsolve(lambda X: logR - poly(X), x0, xtol=1e-14, maxfev=10000)[0]

    # derivative d(logR)/dX = d(poly)/dX
    x = X - O0
    dpoly_dX = c1 + 2*c2*x + 3*c3*x**2 + 4*c4*x**3

    # statistical σ on X from flux errors
    sigma_X_stat = dlogR / abs(dpoly_dX)

    # add Curti+ intrinsic dispersion in quadrature
    sigma_X = np.sqrt(meta["sigma_int_dex"]**2)

    # linear O/H (optional)
    OH = 10**(X - 12)
    # small-sigma symmetric approx in linear space:
    OH_err_sym = OH * np.log(10) * sigma_X

    # warn if outside valid calibration range
    lo, hi = meta["range"]
    out_of_range = not (lo <= X <= hi)

    return {
        "diag": which,
        "X": X, "sigma_X": sigma_X,
        "logR": logR, "dlogR": dlogR,
        "OH": OH, "OH_err_sym": OH_err_sym,
        "valid_range": (lo, hi),
        "out_of_range": out_of_range
    }

# ---- Example usage with your "fluxes" dict (values, errors) ----
res = invert_curti17(fluxes, which="O3N2", x0=8.6)  # tweak x0 if needed
if res["out_of_range"]:
    print(f"Warning: solution {res['X']:.3f} dex is outside Curti+17 {res['diag']} valid range {res['valid_range']}.")

print(f"(12+log O/H) [{res['diag']}] = {res['X']:.3f} ± {res['sigma_X']:.3f} dex")
print(f"O/H ≈ {res['OH']:.3e} ± {res['OH_err_sym']:.3e}  (symmetric approx)")

# ---- Relative to solar (Asplund+2009 by default) ----
X_sun = 8.69  # 12+log(O/H)_sun (Asplund et al. 2009). Change if you adopt a different solar scale.

delta = res["X"] - X_sun                        # dex offset from solar
ratio = 10**delta                                # (O/H)/(O/H)_sun

# Asymmetric 1σ in linear space from dex uncertainty:
sigma_X = res["sigma_X"]
ratio_plus  = ratio * (10**( sigma_X) - 1)       # upper error
ratio_minus = ratio * (1 - 10**(-sigma_X))       # lower error

print(f"(O/H)/(O/H)_sun = {ratio:.3f} +{ratio_plus:.3f}/-{ratio_minus:.3f}  (X_sun = {X_sun:.2f} dex)")

In [ ]:
import pandas as pd
df = pd.DataFrame(
    [(line, f, err, lines.get(line, None)) for line, (f, err) in fluxes.items()],
    columns=["Line", "Flux", "Uncertainty", "λ_rest"]
).sort_values("λ_rest").reset_index(drop=True)

# Create LaTeX table
latex_table = r"""\begin{table}[h]
\centering
\begin{tabular}{l c}
\hline
Line & Flux ($10^{-16}$ erg s$^{-1}$ cm$^{-2}$) \\
\hline
"""
for _, row in df.iterrows():
    val = row["Flux"] / 1e-16
    err = row["Uncertainty"] / 1e-16
    latex_table += f"{row['Line']} & {val:.2f} $\\pm$ {err:.2f} \\\\\n"
latex_table += r"""\hline
\end{tabular}
\caption{Measured emission line fluxes for the star-forming region.}
\end{table}
"""

print(latex_table)

In [ ]:
y = np.log10(fluxes['[NII]-6583']/(fluxes['[SII]-6732']+fluxes['[SII]-6716']))+0.264*np.log10(fluxes['[NII]-6583']/fluxes['H-a'])

print(10**(8.77+y-8.69))

In [ ]:

_O3N2 = (fluxes['[OIII]-5008']/(fluxes['H-b'])) / (fluxes['[NII]-6583']/fluxes['H-a'])
O3N2 = lambda x: np.log10(_O3N2) - (0.281 - 4.765 * x - 2.268 * x**2)
value     = fsolve(O3N2, -0.1, maxfev=10000, xtol=1e-14)

print(10**value[0])

In [ ]:
x_center, y_center = 320,50
radius = 10  # Radius of the aperture in pixels

s = cube.subcube_circle_aperture((x_center, y_center), radius,unit_center=None, unit_radius=None).sum(axis=(1,2))
wavelength = s.wave.coord()  # Wavelength array
flux = s.data           # Flux array

# Plot the spectrum
plt.figure(figsize=(8, 5))
plt.plot(wavelength/(1+z), flux, label='Extracted Spectrum')
#plot a line at the rest wavelength of H-alpha, H-beta, O III and N II
plt.axvline(H_alpha, color='red', linestyle='--', label='Rest Wavelength of H-alpha',alpha=0.5)
plt.axvline(4861.33, color='blue', linestyle='--', label='Rest Wavelength of H-beta',alpha=0.5)
plt.axvline(5006.84, color='green', linestyle='--', label='Rest Wavelength of OIII',alpha=0.5)
plt.axvline(6583.4, color='purple', linestyle='--', label='Rest Wavelength of NII',alpha=0.5)

plt.xlabel('Rest Wavelength (Angstrom)')
plt.ylabel('Flux')
plt.legend()
plt.title('Spectrum from Circular Aperture')
plt.show()

In [ ]:
x_center, y_center = 198,163
radius = 10  # Radius of the aperture in pixels

s = cube.subcube_circle_aperture((x_center, y_center), radius,unit_center=None, unit_radius=None).sum(axis=(1,2))
wavelength = s.wave.coord()  # Wavelength array
flux = s.data           # Flux array

# Plot the spectrum
plt.figure(figsize=(8, 5))
plt.plot(wavelength/(1+z), flux*1e-20, label='Extracted Spectrum')
#plot a line at the rest wavelength of H-alpha, H-beta, O III and N II
plt.axvline(H_alpha, color='red', linestyle='--', label='Rest Wavelength of H-alpha',alpha=0.5)
plt.axvline(4861.33, color='blue', linestyle='--', label='Rest Wavelength of H-beta',alpha=0.5)
plt.axvline(5006.84, color='green', linestyle='--', label='Rest Wavelength of OIII',alpha=0.5)
plt.axvline(6583.4, color='purple', linestyle='--', label='Rest Wavelength of NII',alpha=0.5)

plt.xlabel('Rest Wavelength (Angstrom)')
plt.ylabel('Flux')
plt.legend()
plt.title('Spectrum from Circular Aperture')
plt.show()

In [ ]:
x_center, y_center = 69 ,80
radius = 10  # Radius of the aperture in pixels

s = cube.subcube_circle_aperture((x_center, y_center), radius,unit_center=None, unit_radius=None).mean(axis=(1,2))
wavelength = s.wave.coord()  # Wavelength array
flux = s.data           # Flux array

# Plot the spectrum
plt.figure(figsize=(8, 5))
plt.plot(wavelength/(1+z), flux, label='Extracted Spectrum')
plt.xlabel('Rest Wavelength (Angstrom)')
plt.ylabel('Flux')
plt.legend()
plt.title('Spectrum from Circular Aperture')
plt.show()

In [ ]:
header

In [ ]:
lambda_obs = H_alpha * (1 + redshift)
image_halpha       = cube.bandpass_image(np.linspace(lambda_obs, lambda_obs+20), np.linspace(lambda_obs, lambda_obs+20)+1) / 20
image_halpha_left  = cube.bandpass_image(np.linspace(lambda_obs-30, lambda_obs-10), np.linspace(lambda_obs-30, lambda_obs-10)+1) / 20 / 2
image_halpha_right = cube.bandpass_image(np.linspace(lambda_obs+30, lambda_obs+50), np.linspace(lambda_obs+30, lambda_obs+50)+1) / 20 / 2

In [ ]:
from matplotlib import colors, cm
from matplotlib.colors import LogNorm
from astropy import units as u
from mpl_toolkits.axes_grid1 import make_axes_locatable


plt.figure(figsize=(9, 9))
ax=plt.subplot(111)

zscale = ZScaleInterval()
vmin, vmax = zscale.get_limits((image_halpha - (image_halpha_left+image_halpha_right)).data)

pos=ax.imshow(((image_halpha - (image_halpha_left+image_halpha_right)).data),     origin='lower', 
    cmap='gray',  # Black-and-white colormap
    vmin=0, 
    vmax=vmax,  # ZScale limits
    interpolation='nearest'
)

plt.show()


In [ ]:
vmax